In [1]:
import json
import pandas as pd
import spacy
from collections import Counter

### Load the Training Dataset

In [14]:
with open("../cleaned_data/train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)

print(len(train_data))
print(train_data[0].keys())

4552
dict_keys(['pmid', 'split', 'text', 'tokens', 'labels', 'spans'])


### Check the .json file

In [3]:
print("PMID:", train_data[0]["pmid"])
print("Split:", train_data[0]["split"])
print("Text preview:\n", train_data[0]["text"])
print("Tokens:", train_data[0]["tokens"])
print("Labels sample:", train_data[0]["labels"])
print("Spans sample:", train_data[0]["spans"])

PMID: 7010854
Split: train
Text preview:
 Clinical evaluation of the antihypertensive effect of metoprolol in combination with hydrochlorothiazide and hydralazine in an unselected hypertensive population. One hundred nineteen patients with essential hypertension (96 completing six months and 92 a one year study period) were randomized into four parallel groups and treated with one of four programs: 200 mg of metoprolol plus placebo; 200 mg of metoprolol plus 25 mg of hydrochlorothiazide; 200 mg of metoprolol plus 50 mg hydrochlorothiazide, or; 200 mg metoprolol plus 50 mg of hydralazine. Blood pressure reduction was significant in these all groups and no differences were observed in blood pressure reduction among the groups. During the one year therapy the levels of serum bilirubin, uric acid and triglycerides were significantly increased in all groups but the group treated with metoprolol and hydralazine. Serum cholesterol level did not increase in any group during the one year therap

### Find triggers/keywords
Because the gold labels were given to us, and we have little to no domain knowledge. Token frequency in the document will be conducted to help determine them.

### Start by filtering each token into PIO lists

In [19]:
P_tokens = []
I_tokens = []
O_tokens = []

for doc in train_data:
    tokens = doc["tokens"]
    labels = doc["labels"]

    P = labels.get("participants", [])
    I = labels.get("interventions", labels.get("intervention", []))
    O = labels.get("outcomes", [])

    P_tokens.extend([t.lower() for t, l in zip(tokens, P) if l == 1])
    I_tokens.extend([t.lower() for t, l in zip(tokens, I) if l == 1])
    O_tokens.extend([t.lower() for t, l in zip(tokens, O) if l == 1])

### Clean the tokens

In [23]:
nlp = spacy.load("en_core_web_sm")
stopwords = nlp.Defaults.stop_words

def clean_tokens(tokens):
    cleaned = []
    
    for t in tokens:
        t = t.lower()
        
        if t in stopwords:
            continue
        if not t.isalpha():   
            continue
        
        cleaned.append(t)
    
    return cleaned

P_tokens_clean = clean_tokens(P_tokens)
I_tokens_clean = clean_tokens(I_tokens)
O_tokens_clean = clean_tokens(O_tokens)

### Check frequency of cleaned tokens

In [24]:
P_freq = Counter(P_tokens_clean)
I_freq = Counter(I_tokens_clean)
O_freq = Counter(O_tokens_clean)

print("Top P tokens:", P_freq.most_common(50))
print("Top I tokens:", I_freq.most_common(50))
print("Top O tokens:", O_freq.most_common(50))

Top P tokens: [('patients', 6124), ('children', 1639), ('cancer', 936), ('group', 911), ('years', 904), ('women', 882), ('autism', 802), ('subjects', 716), ('study', 587), ('age', 571), ('n', 513), ('healthy', 471), ('disease', 470), ('treatment', 464), ('randomized', 441), ('chronic', 398), ('undergoing', 396), ('disorder', 369), ('surgery', 366), ('acute', 363), ('spectrum', 361), ('participants', 355), ('adults', 344), ('men', 326), ('trial', 324), ('asd', 321), ('aged', 313), ('randomly', 297), ('treated', 296), ('total', 295), ('mean', 294), ('disorders', 289), ('enrolled', 277), ('control', 273), ('breast', 271), ('groups', 270), ('volunteers', 256), ('received', 248), ('heart', 239), ('autistic', 239), ('advanced', 236), ('coronary', 225), ('syndrome', 222), ('type', 216), ('primary', 210), ('months', 209), ('male', 209), ('pain', 207), ('included', 202), ('therapy', 200)]
Top I tokens: [('placebo', 1742), ('therapy', 997), ('group', 862), ('intervention', 635), ('mg', 627), ('t

Using the frequency of tokens in the code block above, we can identify the following triggers.

From the frequency list for PIO tokens, we selected words that refer directly to the patients, interventions, and outcomes.

Certain words, that may be frequent in one token list may be omitted due to having overlap with another. (e.g., group, or blood)

In [ ]:
P_cues = {
    "patients", "children", "women", "subjects", "participants", "adults", "men", "volunteers"
}

I_cues = {
    "placebo", "therapy", "intervention", "treatment", "received", "receive", "randomized",
    "plus", "versus", "control", "standard", "conventional", "combined", "chemotherapy", "radiotherapy", 
    "surgery", "exercise", "training", "diet", "program", "combination", "oral", "intravenous", "infusion", 
    "supplementation", "vaccine", "vaccination", "dose", "daily"
}

O_cues = {
    "rate", "pain", "levels", "survival", "pressure", "efficacy", "response", "effects", "scores", "rates",
    "symptoms", "adverse", "scale", "safety", "function", "score", "quality", "effect", "life", "duration", 
    "risk", "level", "changes", "mortality"
}